Amplicon sequencing analysis for B. pseudocatenulatum-treated mice

Amplicon sequence reads were processed using QIIME 2 (version 2021.11.0). Reads with a quality score of < 33 and chimeric sequences were filtered out, and representative sequences were generated using the DADA2 denoise-paired plugin (version 2021.11.0). Taxonomic assignment of representative sequences was performed using the Silva database (version 13.8) by training a naive Bayes classifier via the q2-feature-classifier plugin. For amplicon sequence variants (ASVs) identified as Bifidobacterium, we retrieved the 16S rRNA sequence of B. pseudocatenulatum to perform BLASTn analysis.


In [ ]:
source activate qiime2-2021.11

qiime tools import \
  --type 'SampleData[PairedEndSequencesWithQuality]' \
  --input-path manifest_file.tsv \
  --output-path paired-end-demux.qza \
  --input-format PairedEndFastqManifestPhred33V2

qiime demux summarize \
  --i-data paired-end-demux.qza \
  --o-visualization output_file/demux-summary.qzv

qiime dada2 denoise-paired \
  --i-demultiplexed-seqs paired-end-demux.qza \
  --p-trim-left-f 25 \
  --p-trim-left-r 26 \
  --p-trunc-len-f 265 \
  --p-trunc-len-r 230 \
  --o-table table.qza \
  --p-n-threads 0 \
  --o-representative-sequences rep-seqs.qza \
  --o-denoising-stats denoising-stats.qza

qiime metadata tabulate \
  --m-input-file denoising-stats.qza \
  --o-visualization output_file/denoising-stats.qzv

qiime feature-table summarize \
  --i-table table.qza \
  --o-visualization output_file/table.qzv \
  --m-sample-metadata-file sample-metadata.tsv
  
qiime feature-table tabulate-seqs \
  --i-data rep-seqs.qza \
  --o-visualization output_file/rep-seqs.qzv

qiime feature-classifier classify-sklearn \
  --i-classifier /data/home/quj_lab/luxiaoyong/software/silva_database/silva-138-99-nb-classifier.qza \
  --i-reads rep-seqs.qza \
  --o-classification rep-seqs.taxonomy.qza

qiime taxa barplot \
  --i-table rep-seqs.taxonomy.qza \
  --i-taxonomy rep-seqs.taxonomy.qza \
  --m-metadata-file sample-metadata.tsv \
  --o-visualization rep-seqs.taxa-bar-plots.qzv